# SPX comparison — reproducible core-model study

This notebook fully retrains models from the treated SPX cube:



In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display
from matplotlib.lines import Line2D
from scipy.stats import norm, wasserstein_distance

from src.other_models.diffusion_cond_ext import denormalize, fit_normalizer, normalize
from src.other_models.iv_ddpm import IVDDPM
from src.other_models.volgan_train import sample_volgan_diverse, train_volgan
from src.volsurface_latentFM.ivs_autoencoder import IVS_AE
from src.volsurface_latentFM.latent_models_variants import LatentFlowMatch


# Reproductibity 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

torch.use_deterministic_algorithms(True, warn_only=True)

# Independent streams
FINAL_AE_SEED = SEED + 100
FINAL_FM_SEED = SEED + 101
FINAL_FM_SAMPLE_SEED = SEED + 102


PAPER_RC = {
    "font.family": "serif",
    "font.serif": [
        "Linux Libertine O",
        "Libertinus Serif",
        "Times New Roman",
        "Times",
        "Nimbus Roman",
        "Liberation Serif",
        "DejaVu Serif",
    ],
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 9,
    "mathtext.fontset": "stix",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.35,
    "lines.markersize": 4.0,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.edgecolor": "0.75",
    "legend.handlelength": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

mpl.rcParams.update(PAPER_RC)
sns.set_theme(context="paper", style="ticks", rc=PAPER_RC)

CUBE_PATH = Path("data/treated/vol_surface_spx.npz")
OUT_DIR = Path("data/treated/selected_retrain")
FIG_DIR = Path("figs/selected_retrain")

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

N_GEN = 5000
EPOCHS = 1200

# Dedicated epoch budgets for the 5-seed VolGAN and Diffusion runs

# VOLGAN_N_EPOCHS is defined locally in the VolGAN training cell (1500).
DIFFUSION_EPOCHS = 1500
PATHS = 50
PATH_LENGTH = 100
CANDIDATE_PATHS = 512

# Always retrain: there is no saved-study or checkpoint fallback.
RUN_LATENT_DIM_HPO = True
LATENT_DIM_CANDIDATES = (4, 6, 8, 10, 12, 16)
LATENT_HPO_SEEDS = (42, 43, 44, 45, 46)

LATENT_HPO_AE_EPOCHS = 1200
LATENT_HPO_FM_EPOCHS = 800
LATENT_HPO_N_GEN = 5000
LATENT_HPO_N_PROJ = 150

LATENT_ARB_WEIGHT = 0.50
LEGACY_ARB_TOL = 1e-4

LATENT_STUDY_CSV = OUT_DIR / "latent_dimension_optuna_trials.csv"
LATENT_SUMMARY_CSV = OUT_DIR / "latent_dimension_optuna_summary.csv"

if not RUN_LATENT_DIM_HPO:
    raise RuntimeError(
        "Checkpoint-independent mode requires RUN_LATENT_DIM_HPO=True."
    )

"""if tuple(LATENT_DIM_CANDIDATES) != (4, 5, 6, 7, 8, 9, 10):
    raise ValueError(
        "LATENT_DIM_CANDIDATES must be exactly (4, 5, 6, 7, 8, 9, 10)."
    )"""

if len(LATENT_HPO_SEEDS) < 2:
    raise ValueError("Use at least two seeds to report mean ± standard deviation.")

if not CUBE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CUBE_PATH}; build it with constructor_surface_cube.py."
    )

existing_checkpoints = sorted(OUT_DIR.glob("*.pt"))
if existing_checkpoints:
    print(
        "Existing checkpoints detected but they will NOT be loaded. "
        "They are output destinations only:"
    )
    for checkpoint_path in existing_checkpoints:
        print("  ", checkpoint_path)


#data
d = np.load(CUBE_PATH, allow_pickle=True)

cube = np.asarray(d["cube"], dtype=np.float32)
m_new = np.asarray(d["m_grid"], dtype=float).ravel()
tenor_days = np.asarray(d["tenor_days"], dtype=float).ravel()
tau_new = tenor_days / 365.0
asof_dates = np.asarray(d["asof_dates"]) if "asof_dates" in d.files else None

# Canonical constructor layout is (moneyness, tenor, dates).
if cube.shape[:2] == (len(m_new), len(tenor_days)):
    surfaces = cube.transpose(2, 1, 0)
elif cube.shape[1:] == (len(tenor_days), len(m_new)):
    surfaces = cube
else:
    raise ValueError(f"Unexpected cube orientation: {cube.shape}")

if np.nanmedian(surfaces) < 2.0:
    surfaces = surfaces * 100.0

real_iv = torch.tensor(
    surfaces,
    dtype=torch.float32,
    device=device,
).unsqueeze(1)

NT, NM = real_iv.shape[-2:]

if (NT, NM) != (len(tau_new), len(m_new)):
    raise ValueError("Surface shape and stored grids do not agree.")

print("Cube:", CUBE_PATH, cube.shape)
print("Training tensor:", tuple(real_iv.shape))
print("Grid:", NT, "tenors x", NM, "moneyness points")
print("Device:", device)
print("HPO seeds:", LATENT_HPO_SEEDS)
print("Figure font policy: 9 pt Linux Libertine with Times-style fallbacks")


In [ ]:
# Optuna study of latent dimensions

AE_BASE_PARAMS = {
    "hidden_enc": (256, 128, 64),
    "hidden_dec": (64, 128, 256),
    "beta": 1e-2,
    "lambda_cal": 3e-2,
    "lambda_but": 2e-3,
}
AE_FINAL_FIT = {
    "batch_size": 64,
    "lr": 2e-3,
    "warmup_epochs": 200,
}
FM_FINAL_FIT = {
    "batch_size": 64,
    "lr": 1e-4,
}


def set_all_seeds(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def sliced_wasserstein(X, Y, n_proj=150, seed=42):
    """Sliced Wasserstein distance for two sample matrices."""
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    if X.ndim != 2 or Y.ndim != 2 or X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Expected two matrices with the same feature dimension, got {X.shape} and {Y.shape}."
        )

    rng = np.random.default_rng(int(seed))
    directions = rng.standard_normal((int(n_proj), X.shape[1]))
    directions /= np.maximum(
        np.linalg.norm(directions, axis=1, keepdims=True),
        1e-12,
    )

    proj_x = np.sort(X @ directions.T, axis=0)
    proj_y = np.sort(Y @ directions.T, axis=0)

    n_common = max(len(proj_x), len(proj_y))
    u_common = np.linspace(0.0, 1.0, n_common)
    u_x = np.linspace(0.0, 1.0, len(proj_x))
    u_y = np.linspace(0.0, 1.0, len(proj_y))

    distances = []
    for j in range(int(n_proj)):
        x_j = np.interp(u_common, u_x, proj_x[:, j])
        y_j = np.interp(u_common, u_y, proj_y[:, j])
        distances.append(np.mean(np.abs(x_j - y_j)))

    return float(np.mean(distances))


def decode_with_ae(ae_model, latent_codes, lo_value, hi_value):
    """Decode latent codes and return IV surfaces in percentage units."""
    decoded_n = (
        ae_model.decode(latent_codes, device=device)
        .reshape(-1, NT, NM)
        .cpu()
    )
    lo_tensor = torch.tensor(float(lo_value), dtype=torch.float32)
    hi_tensor = torch.tensor(float(hi_value), dtype=torch.float32)
    return denormalize(decoded_n, lo_tensor, hi_tensor).numpy()


def pointwise_quantile_error(real_surfaces, generated_surfaces, quantile):
    """W1 on quantile surfaces."""
    q_real = np.percentile(real_surfaces, quantile, axis=0) * 100
    q_generated = np.percentile(generated_surfaces, quantile, axis=0) * 100
    return float(wasserstein_distance(q_real.ravel(), q_generated.ravel()))


def legacy_phi_batch(surfaces_fraction):
    """
    Historical arbitrage score used in the earlier model comparisons.

    This diagnostic deliberately uses the old linear comparison axes,
    old relative-call transformation and unscaled butterfly second difference.
    It is used only for the Optuna score/reporting in this paper notebook.
    """
    sigma = np.asarray(surfaces_fraction, dtype=float)
    if sigma.ndim != 3 or sigma.shape[1:] != (NT, NM):
        raise ValueError(
            f"Expected surfaces with shape (n, {NT}, {NM}), got {sigma.shape}."
        )

    m_legacy = np.linspace(float(np.min(m_new)), float(np.max(m_new)), NM)
    tau_legacy = np.linspace(float(np.min(tau_new)), float(np.max(tau_new)), NT)

    T = tau_legacy[None, :, None]
    M = m_legacy[None, None, :]
    vol = np.clip(sigma, 1e-8, None)
    T = np.clip(T, 1e-12, None)

    d1 = (-M + 0.5 * vol**2 * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    calls = norm.cdf(d1) - np.exp(M) * norm.cdf(d2)
    calls = np.clip(calls, 1e-12, None)

    dm = np.diff(m_legacy)
    dt = np.diff(tau_legacy)

    dC_dm = np.diff(calls, axis=2) / dm[None, None, :]
    p_call = np.maximum(dC_dm, 0.0).sum(axis=(1, 2))

    d2C = calls[:, :, 2:] - 2.0 * calls[:, :, 1:-1] + calls[:, :, :-2]
    p_bfly = np.maximum(-d2C, 0.0).sum(axis=(1, 2))

    dC_dt = np.diff(calls, axis=1) / dt[None, :, None]
    p_calendar = np.maximum(-dC_dt, 0.0).sum(axis=(1, 2))

    return p_call + p_bfly + p_calendar


# HPO uses the full sample for both fitting and scoring 

n_total = len(real_iv)

real_hpo_train = real_iv
real_hpo_val = real_iv

lo_hpo, hi_hpo = fit_normalizer(real_hpo_train)
surf_hpo_train_n = normalize(real_hpo_train, lo_hpo, hi_hpo)
surf_hpo_val_n = normalize(real_hpo_val, lo_hpo, hi_hpo)
lo_hpo_v = float(lo_hpo.item())
hi_hpo_v = float(hi_hpo.item())

real_hpo_val_fraction = (
    real_hpo_val[:, 0].detach().cpu().numpy() / 100.0
).astype(np.float32)

print(
    "Latent-dimension HPO:",
    f"train={len(real_hpo_train)}",
    f"eval={len(real_hpo_val)}",
)


def latent_dimension_objective(trial):
    z_dim = int(trial.suggest_categorical("z_dim", LATENT_DIM_CANDIDATES))
    trial_seed = int(trial.suggest_categorical("trial_seed", LATENT_HPO_SEEDS))
    set_all_seeds(trial_seed)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ae_trial = IVS_AE(
        z_dim=z_dim,
        hidden_enc=AE_BASE_PARAMS["hidden_enc"],
        hidden_dec=AE_BASE_PARAMS["hidden_dec"],
        beta=AE_BASE_PARAMS["beta"],
        lambda_cal=AE_BASE_PARAMS["lambda_cal"],
        lambda_but=AE_BASE_PARAMS["lambda_but"],
    )
    ae_trial.fit(
        surf_hpo_train_n,
        m_new,
        tau_new,
        surf_val_n=surf_hpo_val_n,
        n_epochs=LATENT_HPO_AE_EPOCHS,
        batch_size=64,
        lr=2e-3,
        warmup_epochs=min(80, max(1, LATENT_HPO_AE_EPOCHS // 3)),
        patience=LATENT_HPO_AE_EPOCHS + 1,
        print_every=10**9,
        lo=lo_hpo_v,
        hi=hi_hpo_v,
        device=device,
    )
    ae_trial.eval()

    with torch.no_grad():
        z_hpo_train = ae_trial.encode(
            surf_hpo_train_n,
            device=device,
            use_mean=True,
        )
        z_hpo_val = ae_trial.encode(
            surf_hpo_val_n,
            device=device,
            use_mean=True,
        )

    fm_trial = LatentFlowMatch(z_dim=z_dim, path="trigo")
    fm_trial.fit(
        z_hpo_train,
        n_epochs=LATENT_HPO_FM_EPOCHS,
        batch_size=64,
        lr=1e-4,
        print_every=max(100, LATENT_HPO_FM_EPOCHS + 1),
        device=device,
    )

    n_eval = LATENT_HPO_N_GEN
    generated_pct = decode_with_ae(
        ae_trial,
        fm_trial.sample(n_eval, device=device),
        lo_hpo_v,
        hi_hpo_v,
    )
    generated_fraction = (generated_pct / 100.0).astype(np.float32)
    real_eval = real_hpo_val_fraction[:]

    reconstructed_pct = decode_with_ae(
        ae_trial,
        z_hpo_val,
        lo_hpo_v,
        hi_hpo_v,
    )
    reconstructed_fraction = (reconstructed_pct / 100.0).astype(np.float32)

    reconstruction_mae = float(
        np.mean(np.abs(reconstructed_fraction - real_hpo_val_fraction))
    )
    global_w1 = float(
        wasserstein_distance(
            real_eval.reshape(-1),
            generated_fraction.reshape(-1),
        )
    )
    swd = sliced_wasserstein(
        real_eval.reshape(len(real_eval), -1),
        generated_fraction.reshape(n_eval, -1),
        n_proj=LATENT_HPO_N_PROJ,
        seed=trial_seed,
    )

    q05 = pointwise_quantile_error(real_eval, generated_fraction, 5)
    q25 = pointwise_quantile_error(real_eval, generated_fraction, 25)
    q50 = pointwise_quantile_error(real_eval, generated_fraction, 50)
    q75 = pointwise_quantile_error(real_eval, generated_fraction, 75)
    q95 = pointwise_quantile_error(real_eval, generated_fraction, 95)
    q99 = pointwise_quantile_error(real_eval, generated_fraction, 99)

    phi = legacy_phi_batch(generated_fraction)
    mean_phi = float(np.mean(phi))
    zero_arb_pct = float(100.0 * np.mean(np.abs(phi) < LEGACY_ARB_TOL))

    distribution_score = (
        0.30 * swd
        + 0.15 * global_w1
        + 0.10 * q05
        + 0.00 * q25
        + 0.10 * q50
        + 0.00 * q75
        + 0.15 * q95
        + 0.20 * q99
        + 0.00 * reconstruction_mae
    )
    objective_score = (
        (1.0 - LATENT_ARB_WEIGHT) * distribution_score
        + LATENT_ARB_WEIGHT * mean_phi
    )

    metrics = {
        "distribution_score": float(distribution_score),
        "global_w1": global_w1,
        "swd": swd,
        "q05": q05,
        "q25": q25,
        "q50": q50,
        "q75": q75,
        "q95": q95,
        "q99": q99,
        "reconstruction_mae": reconstruction_mae,
        "mean_phi": mean_phi,
        "zero_arb_pct": zero_arb_pct,
    }
    for key, value in metrics.items():
        trial.set_user_attr(key, float(value))

    print(
        f"z={z_dim:>2} seed={trial_seed} score={objective_score:.6f} "
        f"SWD={swd:.6f} W1={global_w1:.6f} "
        f"Q25={q25:.6f} Q75={q75:.6f} "
        f"Q95={q95:.6f} Q99={q99:.6f} "
        f"arb-free={zero_arb_pct:.1f}%"
    )

    del ae_trial, fm_trial, z_hpo_train, z_hpo_val
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return float(objective_score)


if RUN_LATENT_DIM_HPO:
    sampler = optuna.samplers.GridSampler(
        {
            "z_dim": list(LATENT_DIM_CANDIDATES),
            "trial_seed": list(LATENT_HPO_SEEDS),
        },
        seed=SEED,
    )
    latent_study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        study_name="paper_lfm_latent_dimension",
    )
    latent_study.optimize(
        latent_dimension_objective,
        n_trials=len(LATENT_DIM_CANDIDATES) * len(LATENT_HPO_SEEDS),
        n_jobs=1,
        show_progress_bar=True,
    )

    latent_rows = []
    for trial in latent_study.trials:
        if trial.value is None or trial.state != optuna.trial.TrialState.COMPLETE:
            continue
        latent_rows.append(
            {
                "trial": int(trial.number),
                "z_dim": int(trial.params["z_dim"]),
                "seed": int(trial.params["trial_seed"]),
                "score": float(trial.value),
                **{
                    key: float(trial.user_attrs[key])
                    for key in (
                        "distribution_score",
                        "global_w1",
                        "swd",
                        "q05",
                        "q25",
                        "q50",
                        "q75",
                        "q95",
                        "q99",
                        "reconstruction_mae",
                        "mean_phi",
                        "zero_arb_pct",
                    )
                },
            }
        )

    latent_trials_df = (
        pd.DataFrame(latent_rows)
        .sort_values(["z_dim", "seed"])
        .reset_index(drop=True)
    )
    if latent_trials_df.empty:
        raise RuntimeError("The latent-dimension Optuna study produced no completed trial.")

    latent_summary_df = (
        latent_trials_df.groupby("z_dim", as_index=False)
        .agg(
            score_mean=("score", "mean"),
            score_std=("score", "std"),
            global_w1_mean=("global_w1", "mean"),
            global_w1_std=("global_w1", "std"),
            swd_mean=("swd", "mean"),
            swd_std=("swd", "std"),
            q05_mean=("q05", "mean"),
            q05_std=("q05", "std"),
            q25_mean=("q25", "mean"),
            q25_std=("q25", "std"),
            q50_mean=("q50", "mean"),
            q50_std=("q50", "std"),
            q75_mean=("q75", "mean"),
            q75_std=("q75", "std"),
            q95_mean=("q95", "mean"),
            q95_std=("q95", "std"),
            q99_mean=("q99", "mean"),
            q99_std=("q99", "std"),
            reconstruction_mae_mean=("reconstruction_mae", "mean"),
            reconstruction_mae_std=("reconstruction_mae", "std"),
            mean_phi_mean=("mean_phi", "mean"),
            mean_phi_std=("mean_phi", "std"),
            zero_arb_pct_mean=("zero_arb_pct", "mean"),
            zero_arb_pct_std=("zero_arb_pct", "std"),
        )
        .sort_values("z_dim")
        .reset_index(drop=True)
    )
    std_columns = [column for column in latent_summary_df if column.endswith("_std")]
    latent_summary_df[std_columns] = latent_summary_df[std_columns].fillna(0.0)

    latent_trials_df.to_csv(LATENT_STUDY_CSV, index=False)
    latent_summary_df.to_csv(LATENT_SUMMARY_CSV, index=False)

best_row = latent_summary_df.loc[latent_summary_df["score_mean"].idxmin()]
BEST_Z_DIM = int(best_row["z_dim"])

print("\nLatent-dimension summary:")
display(latent_summary_df)
print("\nSelected latent dimension:", BEST_Z_DIM)

# Final full-sample retraining with the selected latent dimension.
set_all_seeds(FINAL_AE_SEED)

AE_CKPT = OUT_DIR / f"ivs_ae_z{BEST_Z_DIM}_spx_full.pt"
FM_CKPT = OUT_DIR / f"latent_fm_trigo_z{BEST_Z_DIM}_spx_full.pt"
AE_CKPT_ALIAS = OUT_DIR / "ivs_ae_spx_full.pt"
FM_CKPT_ALIAS = OUT_DIR / "latent_fm_trigo_spx_full.pt"

lo, hi = fit_normalizer(real_iv)
surf_train_n = normalize(real_iv, lo, hi)
lo_v, hi_v = float(lo.item()), float(hi.item())

ae = IVS_AE(
    z_dim=BEST_Z_DIM,
    hidden_enc=AE_BASE_PARAMS["hidden_enc"],
    hidden_dec=AE_BASE_PARAMS["hidden_dec"],
    beta=AE_BASE_PARAMS["beta"],
    lambda_cal=AE_BASE_PARAMS["lambda_cal"],
    lambda_but=AE_BASE_PARAMS["lambda_but"],
)
ae.fit(
    surf_train_n,
    m_new,
    tau_new,
    surf_val_n=None,
    n_epochs=EPOCHS,
    batch_size=AE_FINAL_FIT["batch_size"],
    lr=AE_FINAL_FIT["lr"],
    warmup_epochs=AE_FINAL_FIT["warmup_epochs"],
    patience=EPOCHS + 1,
    print_every=200,
    lo=lo_v,
    hi=hi_v,
    device=device,
)
ae.save(str(AE_CKPT), surf_lo=lo_v, surf_hi=hi_v)
ae.save(str(AE_CKPT_ALIAS), surf_lo=lo_v, surf_hi=hi_v)

# Continue with the in-memory model. No checkpoint is loaded.
ae.eval()
z_train = ae.encode(surf_train_n, device=device, use_mean=True)

set_all_seeds(FINAL_FM_SEED)
fm = LatentFlowMatch(z_dim=BEST_Z_DIM, path="trigo")
fm.fit(
    z_train,
    n_epochs=EPOCHS,
    batch_size=FM_FINAL_FIT["batch_size"],
    lr=FM_FINAL_FIT["lr"],
    print_every=200,
    device=device,
)
fm.save(str(FM_CKPT))
fm.save(str(FM_CKPT_ALIAS))

vae_lo = torch.tensor(lo_v, dtype=torch.float32)
vae_hi = torch.tensor(hi_v, dtype=torch.float32)


def decode_latent(z):
    x = ae.decode(z, device=device).reshape(-1, NT, NM).cpu()
    return denormalize(x, vae_lo, vae_hi).numpy()


set_all_seeds(FINAL_FM_SAMPLE_SEED)
ivs_lfm = decode_latent(fm.sample(N_GEN, device=device))
print("Final L-FM latent dimension:", BEST_Z_DIM)
print("L-FM generated:", ivs_lfm.shape)


In [ ]:
# Latent-dimension sensitivity tables and figures: mean ± standard deviation.

trial_format = {
    "score": "{:.6f}",
    "distribution_score": "{:.6f}",
    "global_w1": "{:.6f}",
    "swd": "{:.6f}",
    "q05": "{:.6f}",
    "q25": "{:.6f}",
    "q50": "{:.6f}",
    "q75": "{:.6f}",
    "q95": "{:.6f}",
    "q99": "{:.6f}",
    "reconstruction_mae": "{:.6f}",
    "mean_phi": "{:.6f}",
    "zero_arb_pct": "{:.1f}%",
}

summary_format = {
    column: ("{:.1f}%" if "zero_arb_pct" in column else "{:.6f}")
    for column in latent_summary_df.columns
    if column != "z_dim"
}

display(latent_trials_df.style.format(trial_format))
display(latent_summary_df.style.format(summary_format))


def plot_dimension_errorbar(
    mean_column,
    std_column,
    label,
    ylabel,
    title,
    filename,
):
    fig, ax = plt.subplots(figsize=(6.6, 3.8))

    ax.errorbar(
        latent_summary_df["z_dim"],
        latent_summary_df[mean_column],
        yerr=latent_summary_df[std_column],
        marker="o",
        capsize=3,
        label=f"{label}: mean ± 1 std",
    )

    ax.axvline(
        BEST_Z_DIM,
        linestyle="--",
        linewidth=1.0,
        label=f"Selected dimension = {BEST_Z_DIM}",
    )

    ax.set_xticks(list(LATENT_DIM_CANDIDATES))
    ax.set_xlabel("Latent dimension")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best")

    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, bbox_inches="tight")
    plt.show()


plot_dimension_errorbar(
    mean_column="score_mean",
    std_column="score_std",
    label="Optuna objective",
    ylabel="Objective",
    title="Latent-Dimension Selection",
    filename="Latent_dimension_objective.png",
)


fig, ax = plt.subplots(figsize=(6.6, 3.8))

metric_specs = [
    ("global_w1_mean", "global_w1_std", "Global W1"),
    ("swd_mean", "swd_std", "Sliced W1"),
    ("q25_mean", "q25_std", "Q25 surface error"),
    ("q75_mean", "q75_std", "Q75 surface error"),
    ("q95_mean", "q95_std", "Q95 surface error"),
    ("q99_mean", "q99_std", "Q99 surface error"),
]

for mean_column, std_column, label in metric_specs:
    ax.errorbar(
        latent_summary_df["z_dim"],
        latent_summary_df[mean_column],
        yerr=latent_summary_df[std_column],
        marker="o",
        capsize=3,
        label=f"{label}: mean ± 1 std",
    )

ax.axvline(
    BEST_Z_DIM,
    linestyle="--",
    linewidth=1.0,
    label=f"Selected dimension = {BEST_Z_DIM}",
)

ax.set_xticks(list(LATENT_DIM_CANDIDATES))
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Distance")
ax.set_title("Distribution and Tail Quality by Latent Dimension")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(FIG_DIR / "Latent_dimension_metrics.png", bbox_inches="tight")
plt.show()


plot_dimension_errorbar(
    mean_column="zero_arb_pct_mean",
    std_column="zero_arb_pct_std",
    label="Legacy arbitrage-free rate",
    ylabel="Arbitrage-free surfaces (%)",
    title="Arbitrage-Free Rate by Latent Dimension",
    filename="Latent_dimension_arbitrage.png",
)


In [ ]:
# Evaluation helpers for multi-seed VolGAN / Diffusion runs
# metrics (global W1, sliced W1, per-quantile W1, legacy arbitrage score, and factors)

QUANTILE_LEVELS_MS = [5, 25, 50, 75, 95, 99]

def _atm_factor_indices():
    atm_idx   = int(np.argmin(np.abs(np.log(m_new))))
    left_idx  = slice(None, NM // 4)
    right_idx = slice(3 * NM // 4, None)
    short_idx = slice(None, NT // 4)
    long_idx  = slice(3 * NT // 4, None)
    return atm_idx, left_idx, right_idx, short_idx, long_idx

_ATM_IDX, _LEFT_IDX, _RIGHT_IDX, _SHORT_IDX, _LONG_IDX = _atm_factor_indices()


def surface_factors_ms(arr):
    """arr: (N, NT, NM) in fraction units. Returns dict of 1-D (N,) factor arrays."""
    atm         = arr[:, :, _ATM_IDX]
    level       = atm.mean(axis=1)
    term_struct = atm[:, _LONG_IDX].mean(axis=1) - atm[:, _SHORT_IDX].mean(axis=1)
    left_wing   = arr[:, :, _LEFT_IDX].mean(axis=(1, 2))
    right_wing  = arr[:, :, _RIGHT_IDX].mean(axis=(1, 2))
    skew        = left_wing - right_wing
    curvature   = (left_wing + right_wing) / 2.0 - level
    return {"Level": level, "Skew": skew, "Term Structure": term_struct, "Curvature": curvature}


REAL_FRAC_MS = real_iv[:, 0].detach().cpu().numpy() / 100.0


def evaluate_generated_full(real_frac, gen_pct, swd_seed=SEED):
    """
    real_frac : (N_real, NT, NM) fraction — ground truth
    gen_pct   : (N_gen,  NT, NM) percentage points — model output

    Returns a flat dict with every metric reported elsewhere in this notebook:
    global W1, sliced W1, per-quantile W1, legacy arbitrage score, and the
    four economic-factor W1 distances.
    """
    gen_frac  = np.asarray(gen_pct, dtype=np.float32) / 100.0
    real_frac = np.asarray(real_frac, dtype=np.float32)

    out = {}
    out["global_w1"] = float(
        wasserstein_distance(real_frac.reshape(-1), gen_frac.reshape(-1))
    )
    out["sliced_w1"] = float(
        sliced_wasserstein(
            real_frac.reshape(len(real_frac), -1),
            gen_frac.reshape(len(gen_frac), -1),
            n_proj=300,
            seed=swd_seed,
        )
    )

    for q in QUANTILE_LEVELS_MS:
        out[f"q{q}"] = float(
            wasserstein_distance(
                np.percentile(real_frac * 100, q, axis=0).ravel(),
                np.percentile(gen_frac * 100, q, axis=0).ravel(),
            )
        )

    phi = legacy_phi_batch(gen_frac.astype(np.float64))
    out["mean_phi"]     = float(np.mean(phi))
    out["zero_arb_pct"] = float(100.0 * np.mean(np.abs(phi) < LEGACY_ARB_TOL))

    real_factors = surface_factors_ms(real_frac)
    gen_factors  = surface_factors_ms(gen_frac)
    for f in ["Level", "Skew", "Term Structure", "Curvature"]:
        out[f"w1_{f.lower().replace(' ', '_')}"] = float(
            wasserstein_distance(real_factors[f], gen_factors[f])
        )

    return out


def summarize_seeds(rows_df, group_cols=("variant",)):
    """Mean +/- std across seeds for every numeric metric column."""
    metric_cols = [c for c in rows_df.columns if c not in list(group_cols) + ["seed"]]
    agg     = {c: ["mean", "std"] for c in metric_cols}
    summary = rows_df.groupby(list(group_cols)).agg(agg)
    summary.columns = [f"{c}_{stat}" for c, stat in summary.columns]
    return summary.reset_index()


def _best_seed(df, variant_name, metric="sliced_w1"):
    """Seed with the lowest value of `metric` (lower = better) for a variant."""
    sub = df[df["variant"] == variant_name]
    return int(sub.loc[sub[metric].idxmin(), "seed"])


In [ ]:
# IV-DDPM (Diffusion)
DIFFPEN_SEEDS = (42, 43, 44, 45, 46)

diffpen_seed_rows      = []
diffpen_by_seed        = {}
diffpen_model_by_seed  = {}

for dp_seed in DIFFPEN_SEEDS:
    set_all_seeds(dp_seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    diffpen_s = IVDDPM(
        tau_grid=tau_new,
        lo=lo_v,
        hi=hi_v,
        T=500,
        enc_channels=32,
        bot_channels=64,
        emb_dim=128,
        lambda_smile=0.01,
        lambda_ttm=0.01,
        device=device,
    )
    diffpen_s.fit(
        surf_train_n,
        surf_val_n=None,
        n_epochs=DIFFUSION_EPOCHS,
        batch_size=64,
        lr=3e-4,
        print_every=DIFFUSION_EPOCHS + 1,
        patience=DIFFUSION_EPOCHS + 1,
    )

    gen_s = diffpen_s.sample(N_GEN, sampler="ddim", ddim_steps=100)
    m_s   = evaluate_generated_full(REAL_FRAC_MS, gen_s, swd_seed=dp_seed)

    diffpen_seed_rows.append({"variant": "Diffusion", "seed": dp_seed, **m_s})
    diffpen_by_seed[dp_seed]       = gen_s
    diffpen_model_by_seed[dp_seed] = diffpen_s

    print(f"[Diffusion seed={dp_seed}]  "
          f"global_w1={m_s['global_w1']:.5f}  swd={m_s['sliced_w1']:.5f}  "
          f"arb-free={m_s['zero_arb_pct']:.1f}%")

diffpen_seed_df = pd.DataFrame(diffpen_seed_rows)

BEST_DP_SEED = _best_seed(diffpen_seed_df, "Diffusion", metric="sliced_w1")
ivs_diffpen  = diffpen_by_seed[BEST_DP_SEED]
diffpen      = diffpen_model_by_seed[BEST_DP_SEED]

diffpen.save(str(OUT_DIR / "iv_ddpm_best.pt"))
print(f"\nBest Diffusion seed: {BEST_DP_SEED}  "
      f"(swd={diffpen_seed_df.set_index('seed').loc[BEST_DP_SEED, 'sliced_w1']:.5f})")

diffpen_summary_df = summarize_seeds(diffpen_seed_df, group_cols=("variant",))
print("\nDiffusion — mean +/- std across seeds:")
display(diffpen_summary_df)

diffpen_seed_df.to_csv(OUT_DIR / "diffusion_multiseed_metrics.csv", index=False)
diffpen_summary_df.to_csv(OUT_DIR / "diffusion_multiseed_summary.csv", index=False)


In [ ]:
torch.manual_seed(50)
N_GEN = 5000
AE_CKPT_ALIAS = OUT_DIR / "ivs_ae_spx_full.pt"
FM_CKPT_ALIAS = OUT_DIR / "latent_fm_trigo_spx_full.pt"

if not AE_CKPT_ALIAS.exists() or not FM_CKPT_ALIAS.exists():
    raise FileNotFoundError(
        f"Missing checkpoint(s): {AE_CKPT_ALIAS} / {FM_CKPT_ALIAS}. "
        "Run the training cell above at least once first."
    )

ae = IVS_AE.load(str(AE_CKPT_ALIAS), device=device)
ae.to(device)
ae.eval()

fm = LatentFlowMatch.load(str(FM_CKPT_ALIAS), device=device)

BEST_Z_DIM = ae.z_dim
lo_v, hi_v = float(ae.surf_lo), float(ae.surf_hi)
vae_lo = torch.tensor(lo_v, dtype=torch.float32)
vae_hi = torch.tensor(hi_v, dtype=torch.float32)


def decode_latent(z):
    x = ae.decode(z, device=device).reshape(-1, NT, NM).cpu()
    return denormalize(x, vae_lo, vae_hi).numpy()


ivs_lfm = decode_latent(fm.sample(N_GEN, device=device))
print("Loaded AE:", AE_CKPT_ALIAS)
print("Loaded FM:", FM_CKPT_ALIAS)
print("L-FM latent dimension:", BEST_Z_DIM)
print("L-FM generated:", ivs_lfm.shape)


In [ ]:
#  VolGAN: train on 5 seeds 
from src.other_models.VolGan import (
    Generator,
    Discriminator,
    arbitrage_penalty_tensor,
    penalty_mutau_tensor,
    smallBS_tensor,
)

VOLGAN_SEEDS         = (42, 43, 44, 45, 46)
VOLGAN_NOISE_DIM     = 32
VOLGAN_HIDDEN_DIM    = 128
VOLGAN_N_EPOCHS      = 1500
VOLGAN_N_GRAD        = 100
VOLGAN_LR            = 1e-4
VOLGAN_BATCH_SIZE    = 32
VOLGAN_REWEIGHT_BETA = 3.0            # exponential-tilting strength
N_POOL_VG            = max(N_GEN, 20000)

# ── Flatten training surfaces + consecutive pairs (fixed, seed-independent) ──
NT_g    = surf_train_n.shape[2]
NM_g    = surf_train_n.shape[3]
Ng_vg   = NT_g * NM_g
flat_tr = surf_train_n.squeeze(1).reshape(-1, Ng_vg)

cond_vg   = flat_tr[:-1]
target_vg = flat_tr[1:] - flat_tr[:-1]
N_tr_vg   = cond_vg.shape[0]
print(f"VolGAN data: {N_tr_vg} pairs, surface dim={Ng_vg} "
      f"({NT_g} tenors x {NM_g} moneyness)")


def make_diff_mats(NT, NM, dev):
    Ng = NT * NM
    m_mat = torch.zeros(NT * (NM - 1), Ng, device=dev)
    for t in range(NT):
        for k in range(NM - 1):
            r = t * (NM - 1) + k
            m_mat[r, t * NM + k]     = -1.0
            m_mat[r, t * NM + k + 1] =  1.0
    t_mat = torch.zeros((NT - 1) * NM, Ng, device=dev)
    for t in range(NT - 1):
        for k in range(NM):
            r = t * NM + k
            t_mat[r,  t * NM + k]      = -1.0
            t_mat[r, (t + 1) * NM + k] =  1.0
    return m_mat, t_mat


m_mat_vg, t_mat_vg = make_diff_mats(NT_g, NM_g, device)


def smooth_loss(surf_batch, m_mat, t_mat):
    pm = ((surf_batch @ m_mat.T) ** 2).mean()
    pt = ((surf_batch @ t_mat.T) ** 2).mean()
    return pm, pt


def exponential_tilt_reweight(surfaces, phi, n_output, beta=3.0, seed=None):
    """Weighted Monte Carlo resampling via exponential tilting."""
    surfaces = np.asarray(surfaces, dtype=np.float32)
    phi      = np.asarray(phi, dtype=float).reshape(-1)
    rng      = np.random.default_rng(None if seed is None else int(seed))

    log_w  = -beta * phi
    log_w -= log_w.max()
    w      = np.exp(log_w)
    w     /= w.sum()

    selected = rng.choice(len(surfaces), size=int(n_output), replace=True, p=w)
    return surfaces[selected], selected


def volgan_arbitrage_phi(surfaces_pct):
    """Exact BS-call-price arbitrage penalty used to drive the reweighting."""
    n_pool      = len(surfaces_pct)
    surfs_t     = torch.tensor(surfaces_pct / 100.0, dtype=torch.float32, device=device)
    surfs_nm_nt = surfs_t.permute(0, 2, 1)

    mP_t, mP_k, mPb_k = penalty_mutau_tensor(m_new, tau_new * 365.0, device)
    m_t   = torch.tensor(m_new,   dtype=torch.float32, device=device)
    tau_t = torch.tensor(tau_new, dtype=torch.float32, device=device)

    with torch.no_grad():
        small_bs = smallBS_tensor(
            m_t.view(1, -1, 1).expand(n_pool, NM, NT),
            tau_t.view(1, 1, -1).expand(n_pool, NM, NT),
            surfs_nm_nt,
            r=0,
        )
        _, _, _, phi = arbitrage_penalty_tensor(
            small_bs,
            mP_t.unsqueeze(0).expand(n_pool, -1, -1),
            mP_k.unsqueeze(0).expand(n_pool, -1, -1),
            mPb_k.unsqueeze(0).expand(n_pool, -1, -1),
        )
    return phi.detach().cpu().numpy()


def sample_volgan_random_real(gen, noise_dim, n_pool, seed):
    """
    Raw generation conditioning each of n_pool samples on an indepenndant
    drawn real training surface - much richer conditioning diversity than
    repeating a handful of fixed seed surfaces.
    """
    rng_idx  = torch.randint(0, N_tr_vg + 1, (n_pool,))
    seed_vec = flat_tr[rng_idx]
    with torch.no_grad():
        noise  = torch.randn(n_pool, noise_dim, device=device)
        fake_n = gen(noise, seed_vec) + seed_vec
    x01  = (fake_n.cpu() + 1.0) / 2.0
    surf = (x01 * (hi_v - lo_v) + lo_v).numpy()
    return surf.reshape(n_pool, NT_g, NM_g)


volgan_seed_rows    = []
volgan_raw_by_seed  = {}
volgan_rw_by_seed   = {}
volgan_gen_by_seed  = {}
volgan_loss_by_seed = {}

for vg_seed in VOLGAN_SEEDS:
    set_all_seeds(vg_seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    vg_gen  = Generator(noise_dim=VOLGAN_NOISE_DIM, cond_dim=Ng_vg,
                        hidden_dim=VOLGAN_HIDDEN_DIM, output_dim=Ng_vg).to(device)
    vg_disc = Discriminator(in_dim=2 * Ng_vg, hidden_dim=VOLGAN_HIDDEN_DIM).to(device)
    opt_g   = torch.optim.RMSprop(vg_gen.parameters(),  lr=VOLGAN_LR)
    opt_d   = torch.optim.RMSprop(vg_disc.parameters(), lr=VOLGAN_LR)
    bce     = torch.nn.BCELoss().to(device)

    # Phase 1: gradient matching 
    BCE_norms, Pm_norms, Pt_norms = [], [], []
    vg_gen.train()
    for _ in range(VOLGAN_N_GRAD):
        perm = torch.randperm(N_tr_vg)
        for i in range(0, N_tr_vg, VOLGAN_BATCH_SIZE):
            idx   = perm[i:i + VOLGAN_BATCH_SIZE]
            cond  = cond_vg[idx]; B = cond.shape[0]
            noise = torch.randn(B, VOLGAN_NOISE_DIM, device=device)
            fake  = vg_gen(noise, cond)
            fake_surf = fake + cond

            opt_g.zero_grad()
            d_fake = vg_disc(torch.cat([cond, fake], dim=-1))
            bce(d_fake, torch.ones_like(d_fake)).backward(retain_graph=True)
            n_bce = sum(p.grad.data.norm(2).item() ** 2
                        for p in vg_gen.parameters() if p.grad is not None) ** 0.5
            BCE_norms.append(n_bce); opt_g.zero_grad()

            pm, pt = smooth_loss(fake_surf, m_mat_vg, t_mat_vg)
            pm.backward(retain_graph=True)
            n_pm = sum(p.grad.data.norm(2).item() ** 2
                       for p in vg_gen.parameters() if p.grad is not None) ** 0.5
            Pm_norms.append(n_pm); opt_g.zero_grad()

            pt.backward()
            n_pt = sum(p.grad.data.norm(2).item() ** 2
                       for p in vg_gen.parameters() if p.grad is not None) ** 0.5
            Pt_norms.append(n_pt); opt_g.zero_grad()

    alpha_vg = float(np.mean(np.array(BCE_norms) / (np.array(Pm_norms) + 1e-12)))
    beta_vg  = float(np.mean(np.array(BCE_norms) / (np.array(Pt_norms) + 1e-12)))

    # Phase 2: main adversarial training 
    gen_losses, disc_losses = [], []
    vg_gen.train(); vg_disc.train()

    for epoch in range(VOLGAN_N_EPOCHS):
        perm = torch.randperm(N_tr_vg)
        gl_ep, dl_ep = [], []
        for i in range(0, N_tr_vg, VOLGAN_BATCH_SIZE):
            idx   = perm[i:i + VOLGAN_BATCH_SIZE]
            cond  = cond_vg[idx]; real = target_vg[idx]; B = cond.shape[0]
            ones  = torch.ones(B, 1, device=device)
            zeros = torch.zeros(B, 1, device=device)

            noise = torch.randn(B, VOLGAN_NOISE_DIM, device=device)
            fake  = vg_gen(noise, cond).detach()
            opt_d.zero_grad()
            d_real = vg_disc(torch.cat([cond, real], dim=-1))
            d_fake = vg_disc(torch.cat([cond, fake], dim=-1))
            d_loss = 0.5 * (bce(d_real, ones) + bce(d_fake, zeros))
            d_loss.backward(); opt_d.step()

            noise  = torch.randn(B, VOLGAN_NOISE_DIM, device=device)
            fake   = vg_gen(noise, cond)
            pm, pt = smooth_loss(fake + cond, m_mat_vg, t_mat_vg)
            d_fake = vg_disc(torch.cat([cond, fake], dim=-1))
            g_loss = bce(d_fake, ones) + alpha_vg * pm + beta_vg * pt
            opt_g.zero_grad(); g_loss.backward(); opt_g.step()

            gl_ep.append(g_loss.item()); dl_ep.append(d_loss.item())

        gen_losses.append(float(np.mean(gl_ep)))
        disc_losses.append(float(np.mean(dl_ep)))

    volgan_loss_by_seed[vg_seed] = {"gen": gen_losses, "disc": disc_losses, "arb": []}

    torch.save({
        "gen": vg_gen.state_dict(), "disc": vg_disc.state_dict(),
        "alpha": alpha_vg, "beta": beta_vg,
        "noise_dim": VOLGAN_NOISE_DIM, "hidden_dim": VOLGAN_HIDDEN_DIM,
        "NT": NT_g, "NM": NM_g, "lo": lo_v, "hi": hi_v,
    }, str(OUT_DIR / f"volgan_seed{vg_seed}.pt"))

    # Raw generation: condition on independently-drawn real surfaces
    raw_pool = sample_volgan_random_real(vg_gen, VOLGAN_NOISE_DIM, N_POOL_VG, vg_seed)

    phi_s = volgan_arbitrage_phi(raw_pool)
    rw_s, rw_idx = exponential_tilt_reweight(
        surfaces=raw_pool, phi=phi_s, n_output=N_GEN,
        beta=VOLGAN_REWEIGHT_BETA, seed=vg_seed,
    )

    raw_eval = raw_pool[:N_GEN]
    m_raw = evaluate_generated_full(REAL_FRAC_MS, raw_eval, swd_seed=vg_seed)
    m_rw  = evaluate_generated_full(REAL_FRAC_MS, rw_s,     swd_seed=vg_seed)

    volgan_seed_rows.append({"variant": "VolGAN (raw)",        "seed": vg_seed, **m_raw})
    volgan_seed_rows.append({"variant": "VolGAN (reweighted)", "seed": vg_seed, **m_rw})

    volgan_raw_by_seed[vg_seed] = raw_eval
    volgan_rw_by_seed[vg_seed]  = rw_s
    volgan_gen_by_seed[vg_seed] = vg_gen

    gl_tail = np.mean(gen_losses[-20:])
    dl_tail = np.mean(disc_losses[-20:])
    print(f"[VolGAN seed={vg_seed}]  "
          f"raw swd={m_raw['sliced_w1']:.5f}  raw mean_phi={m_raw['mean_phi']:.5f}  "
          f"raw IV range=[{raw_eval.min():.2f}%, {raw_eval.max():.2f}%]")
    print(f"    reweighted swd={m_rw['sliced_w1']:.5f}  "
          f"reweighted arb-free={m_rw['zero_arb_pct']:.1f}%")
    print(f"    final-20-epoch G loss={gl_tail:.4f}  D loss={dl_tail:.4f}")

volgan_seed_df = pd.DataFrame(volgan_seed_rows)

BEST_VG_RAW_SEED = _best_seed(volgan_seed_df, "VolGAN (raw)",        metric="sliced_w1")
BEST_VG_RW_SEED  = _best_seed(volgan_seed_df, "VolGAN (reweighted)", metric="sliced_w1")

vg_iv_raw  = volgan_raw_by_seed[BEST_VG_RAW_SEED]
ivs_volgan = volgan_rw_by_seed[BEST_VG_RW_SEED]

print(f"\nBest VolGAN (raw) seed:        {BEST_VG_RAW_SEED}")
print(f"Best VolGAN (reweighted) seed: {BEST_VG_RW_SEED}")

volgan_summary_df = summarize_seeds(volgan_seed_df, group_cols=("variant",))
print("\nVolGAN — mean +/- std across seeds:")
display(volgan_summary_df)

volgan_seed_df.to_csv(OUT_DIR / "volgan_multiseed_metrics.csv", index=False)
volgan_summary_df.to_csv(OUT_DIR / "volgan_multiseed_summary.csv", index=False)


In [ ]:
#  L-FM: full retrain at a fixed latent dimension, on 5 seeds 
def set_all_seeds(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

AE_BASE_PARAMS = {
    "hidden_enc": (256, 128, 64),
    "hidden_dec": (64, 128, 256),
    "beta": 1e-2,
    "lambda_cal": 3e-2,
    "lambda_but": 2e-3,
}
AE_FINAL_FIT = {
    "batch_size": 64,
    "lr": 2e-3,
    "warmup_epochs": 200,
}
FM_FINAL_FIT = {
    "batch_size": 64,
    "lr": 1e-4,
}

LFM_ZDIM  = 6
LFM_SEEDS = (42, 43, 44, 45, 46)

lfm_seed_rows_full  = []
lfm_by_seed_full    = {}
lfm_ae_by_seed_full = {}
lfm_fm_by_seed_full = {}

for lfm_seed in LFM_SEEDS:
    set_all_seeds(lfm_seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ae_s = IVS_AE(
        z_dim=LFM_ZDIM,
        hidden_enc=AE_BASE_PARAMS["hidden_enc"],
        hidden_dec=AE_BASE_PARAMS["hidden_dec"],
        beta=AE_BASE_PARAMS["beta"],
        lambda_cal=AE_BASE_PARAMS["lambda_cal"],
        lambda_but=AE_BASE_PARAMS["lambda_but"],
    )
    ae_s.fit(
        surf_train_n,
        m_new,
        tau_new,
        surf_val_n=None,
        n_epochs=EPOCHS,
        batch_size=AE_FINAL_FIT["batch_size"],
        lr=AE_FINAL_FIT["lr"],
        warmup_epochs=AE_FINAL_FIT["warmup_epochs"],
        patience=EPOCHS + 1,
        print_every=EPOCHS + 1,
        lo=lo_v,
        hi=hi_v,
        device=device,
    )
    ae_s.eval()
    z_train_s = ae_s.encode(surf_train_n, device=device, use_mean=True)

    fm_s = LatentFlowMatch(z_dim=LFM_ZDIM, path="trigo")
    fm_s.fit(
        z_train_s,
        n_epochs=EPOCHS,
        batch_size=FM_FINAL_FIT["batch_size"],
        lr=FM_FINAL_FIT["lr"],
        print_every=EPOCHS + 1,
        device=device,
    )

    def _decode_lfm_seed(z, ae_local=ae_s):
        x = ae_local.decode(z, device=device).reshape(-1, NT, NM).cpu()
        return denormalize(x, vae_lo, vae_hi).numpy()

    gen_s = _decode_lfm_seed(fm_s.sample(N_GEN, device=device))
    m_s   = evaluate_generated_full(REAL_FRAC_MS, gen_s, swd_seed=lfm_seed)

    lfm_seed_rows_full.append({"variant": "L-FM", "seed": lfm_seed, **m_s})
    lfm_by_seed_full[lfm_seed]    = gen_s
    lfm_ae_by_seed_full[lfm_seed] = ae_s
    lfm_fm_by_seed_full[lfm_seed] = fm_s

    print(f"[L-FM seed={lfm_seed}]  "
          f"global_w1={m_s['global_w1']:.5f}  swd={m_s['sliced_w1']:.5f}  "
          f"arb-free={m_s['zero_arb_pct']:.1f}%")

# Overwrite lfm_seed_df / lfm_summary_df / ivs_lfm with the full retrain
# results, so the combined summary cell below uses these instead of the
# HPO-derived (factor-NaN) rows from the cell above.
lfm_seed_df = pd.DataFrame(lfm_seed_rows_full)

BEST_LFM_SEED = _best_seed(lfm_seed_df, "L-FM", metric="sliced_w1")
ivs_lfm = lfm_by_seed_full[BEST_LFM_SEED]
ae      = lfm_ae_by_seed_full[BEST_LFM_SEED]
fm      = lfm_fm_by_seed_full[BEST_LFM_SEED]

ae.save(str(OUT_DIR / f"ivs_ae_z{LFM_ZDIM}_best.pt"), surf_lo=lo_v, surf_hi=hi_v)
fm.save(str(OUT_DIR / f"latent_fm_z{LFM_ZDIM}_best.pt"))

best_lfm_swd = lfm_seed_df.set_index("seed").loc[BEST_LFM_SEED, "sliced_w1"]
print(f"\nBest L-FM seed: {BEST_LFM_SEED}  (z_dim={LFM_ZDIM}, swd={best_lfm_swd:.5f})")

lfm_summary_df = summarize_seeds(lfm_seed_df, group_cols=("variant",))
print(f"\nL-FM (z_dim={LFM_ZDIM}) — mean +/- std across seeds:")
display(lfm_summary_df)

lfm_seed_df.to_csv(OUT_DIR / "lfm_multiseed_metrics.csv", index=False)
lfm_summary_df.to_csv(OUT_DIR / "lfm_multiseed_summary.csv", index=False)


In [ ]:
# Final results table: all models, mean +/- std across seeds 
RESULT_METRICS = [
    ("global_w1",          "Global W1"),
    ("sliced_w1",          "SWD"),
    ("q5",                 "W1 Q5"),
    ("q25",                "W1 Q25"),
    ("q50",                "W1 Q50"),
    ("q75",                "W1 Q75"),
    ("q95",                "W1 Q95"),
    ("q99",                "W1 Q99"),
    ("w1_level",           "W1 Level"),
    ("w1_skew",             "W1 Skew"),
    ("w1_term_structure",   "W1 Term Structure"),
    ("w1_curvature",        "W1 Curvature"),
    ("zero_arb_pct",       "% arb-free"),
]

combined_seed_df    = pd.concat(
    [lfm_seed_df, volgan_seed_df, diffpen_seed_df], ignore_index=True
)
combined_summary_df = summarize_seeds(combined_seed_df, group_cols=("variant",))

MODEL_ORDER = ["L-FM", "VolGAN (raw)", "VolGAN (reweighted)", "Diffusion"]
model_cols  = [m for m in MODEL_ORDER if m in combined_seed_df["variant"].unique()]

def _fmt_cell(mean, std, is_pct=False):
    if pd.isna(mean):
        return "--"
    if pd.isna(std):
        std = 0.0
    if is_pct:
        return f"{mean:.1f} ± {std:.1f}%"
    return f"{mean:.5f} ± {std:.5f}"

final_results_rows = []
for col, label in RESULT_METRICS:
    row = {"Metric": label}
    for model in model_cols:
        sub = combined_seed_df[combined_seed_df["variant"] == model][col]
        mean, std = sub.mean(), sub.std()
        row[model] = _fmt_cell(mean, std, is_pct=(col == "zero_arb_pct"))
    final_results_rows.append(row)

final_results_df = pd.DataFrame(final_results_rows).set_index("Metric")

print("Final results — mean +/- std across seeds (42, 43, 44, 45, 46):")
display(final_results_df)

final_results_df.to_csv(OUT_DIR / "final_results_table.csv")

# Optional LaTeX export for direct inclusion in the paper.
latex_table = final_results_df.to_latex(
    escape=True,
    caption="Distributional and arbitrage metrics across models, "
            "mean $\\pm$ std over 5 seeds (42--46).",
    label="tab:final_results",
)
(OUT_DIR / "final_results_table.tex").write_text(latex_table)
print(f"\nSaved: {OUT_DIR / 'final_results_table.csv'}")
print(f"Saved: {OUT_DIR / 'final_results_table.tex'}")


In [ ]:
# Construct the core-model comparison dictionary.

def frac(x):
    x = np.asarray(x, dtype=np.float32)
    return x / 100.0 if np.nanmedian(x) > 2.0 else x


def model_level_metrics(real, generated, swd_seed=SEED):
    real = np.asarray(real, dtype=np.float32)
    generated = np.asarray(generated, dtype=np.float32)

    return {
        "global_w1": float(
            wasserstein_distance(
                real.reshape(-1),
                generated.reshape(-1),
            )
        ),
        "sliced_w1": float(
            sliced_wasserstein(
                real.reshape(len(real), -1),
                generated.reshape(len(generated), -1),
                n_proj=300,
                seed=swd_seed,
            )
        ),
        "mean_iv": float(generated.mean()),
        "std_iv": float(generated.std()),
    }


real_np = frac(real_iv[:, 0].detach().cpu().numpy())

models_primary = {
    "Data (target)": real_np,
    "L-FM": frac(ivs_lfm),
    "Diffusion": frac(ivs_diffpen),
    "VolGAN": frac(ivs_volgan),
    "VolGAN (raw)": frac(vg_iv_raw),
}

clr = {
    "Data (target)": "#222222",
    "L-FM": "#E07B00",
    "Diffusion": "#1F77B4",
    "VolGAN": "#B41F80",
    "VolGAN (raw)": "#9467BD",
}

ls = {
    "Data (target)": "-",
    "L-FM": "--",
    "Diffusion": "-.",
    "VolGAN": ":",
    "VolGAN (raw)": (0, (5, 2)),
}

# Report reweighting uncertainty across independent resampling seeds.
volgan_seed_metric_rows = []

"""for seed, sample in volgan_reweighted_samples.items():
    metrics = model_level_metrics(
        real_np,
        frac(sample),
        swd_seed=SEED,
    )
    volgan_seed_metric_rows.append(
        {
            "seed": int(seed),
            **metrics,
        }
    )

volgan_seed_metrics_df = pd.DataFrame(volgan_seed_metric_rows)

volgan_seed_summary_df = pd.DataFrame(
    {
        "metric": ["Global W1", "Sliced W1", "Mean IV", "IV standard deviation"],
        "mean": [
            volgan_seed_metrics_df["global_w1"].mean(),
            volgan_seed_metrics_df["sliced_w1"].mean(),
            volgan_seed_metrics_df["mean_iv"].mean(),
            volgan_seed_metrics_df["std_iv"].mean(),
        ],
        "std": [
            volgan_seed_metrics_df["global_w1"].std(ddof=1),
            volgan_seed_metrics_df["sliced_w1"].std(ddof=1),
            volgan_seed_metrics_df["mean_iv"].std(ddof=1),
            volgan_seed_metrics_df["std_iv"].std(ddof=1),
        ],
    }
)

print("VolGAN gentle-reweighting metrics across seeds: mean ± standard deviation")
display(
    volgan_seed_summary_df.style.format(
        {
            "mean": "{:.6f}",
            "std": "{:.6f}",
        }
    )
)"""

# Persist generated arrays before plotting.
payload = {
    "m_grid": m_new,
    "tenor_days": tenor_days,
}

for name, arr in models_primary.items():
    key = "".join(
        character.lower() if character.isalnum() else "_"
        for character in name
    ).strip("_")
    payload[key] = arr.astype(np.float32)

np.savez_compressed(
    OUT_DIR / "core_model_surfaces.npz",
    **payload,
)

for name, arr in models_primary.items():
    print(
        f"{name:<16} {arr.shape} "
        f"mean={arr.mean():.4f} std={arr.std():.4f}"
    )


In [ ]:
# Primary graphs: density, smile and skew.

fig, ax = plt.subplots(figsize=(6.6, 3.8))

for name, arr in models_primary.items():
    sns.kdeplot(
        arr.reshape(-1),
        ax=ax,
        label=name,
        color=clr[name],
        linewidth=1.35,
        linestyle=ls[name],
    )

ax.set_xlim(real_np.min() * 0.85, 0.62)
ax.set_xlabel("Implied volatility")
ax.set_ylabel("Density")
ax.set_title("Implied Volatility Distribution")
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(FIG_DIR / "Models_distribution.png", bbox_inches="tight")
plt.show()


t_indices = [2, NT // 2, NT - 3]

t_labels = [
    rf"Short $\tau={tau_new[t_indices[0]]:.2f}$ yr",
    rf"Medium $\tau={tau_new[t_indices[1]]:.2f}$ yr",
    rf"Long $\tau={tau_new[t_indices[2]]:.2f}$ yr",
]


fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.0, 3.7),
    sharey=True,
)

for panel, (ax, t_idx, label) in enumerate(
    zip(axes, t_indices, t_labels)
):
    for name, arr in models_primary.items():
        ax.plot(
            m_new,
            arr[:, t_idx, :].mean(axis=0),
            label=name,
            color=clr[name],
            linestyle=ls[name],
        )

    ax.set_title(label)
    ax.set_xlabel("Moneyness")

    if panel == 0:
        ax.set_ylabel("Implied volatility")

    ax.legend(loc="best")

fig.suptitle("Average Volatility Smile", y=1.01)

plt.tight_layout()
plt.savefig(FIG_DIR / "Models_smiles.png", bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.0, 3.7),
    sharey=True,
)

for panel, (ax, t_idx, label) in enumerate(
    zip(axes, t_indices, t_labels)
):
    for name, arr in models_primary.items():
        skew = np.gradient(
            arr[:, t_idx, :],
            m_new,
            axis=1,
        ).mean(axis=0)

        ax.plot(
            m_new,
            skew,
            label=name,
            color=clr[name],
            linestyle=ls[name],
        )


    ax.set_title(label)
    ax.set_xlabel("Moneyness")

    if panel == 0:
        ax.set_ylabel(
            r"$\partial \sigma_{\mathrm{IV}}/\partial m$"
        )

    ax.legend(loc="best")

fig.suptitle("Average Volatility Skew", y=1.01)

plt.tight_layout()
plt.savefig(FIG_DIR / "Models_skew.png", bbox_inches="tight")
plt.show()


In [ ]:
# Q-Q plots
q_ref = real_np.reshape(-1)

qq_models = {
    "L-FM": models_primary["L-FM"],
    "VolGAN raw": models_primary["VolGAN (raw)"],
    "Diffusion": models_primary["Diffusion"],
}

qq_colors = {
    "L-FM": "#E07B00",
    "VolGAN raw": "#9467BD",
    "Diffusion": "#1F77B4",
}

percentiles = np.linspace(0.0, 100.0, 500)
q_real = np.percentile(q_ref, percentiles)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(9.0, 3.3),
)

for ax, (name, arr) in zip(axes, qq_models.items()):
    q_model = np.percentile(
        arr.reshape(-1),
        percentiles,
    )

    ax.scatter(
        q_real,
        q_model,
        s=7,
        color=qq_colors[name],
        alpha=0.55,
        label=name,
    )

    lower = min(q_real.min(), q_model.min())
    upper = max(q_real.max(), q_model.max())

    ax.plot(
        [lower, upper],
        [lower, upper],
        color="0.35",
        linewidth=1.0,
        label="Perfect fit",
    )

    ax.set_xlabel("Real quantiles")
    ax.set_ylabel(f"{name} quantiles")
    ax.set_title(f"{name} vs real")
    ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig(FIG_DIR / "Models_QQ.png", bbox_inches="tight")
plt.show()


In [ ]:
# 5th, 95th and 99th-percentile surface figures:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

M, TGRID = np.meshgrid(m_new, tau_new)
cmap = cm.RdYlGn_r

model_legend_handles = [
    Line2D(
        [0],
        [0],
        color=clr[name],
        linewidth=1.5,
        label=name,
    )
    for name in models_primary
]

for quantile in (5, 95, 99):
    quantile_surfaces = {
        name: np.percentile(
            arr * 100.0,
            quantile,
            axis=0,
        )
        for name, arr in models_primary.items()
    }

    all_values = np.concatenate(
        [
            surface.ravel()
            for surface in quantile_surfaces.values()
        ]
    )

    vmin, vmax = np.percentile(
        all_values,
        [1, 99],
    )

    surface_norm = mcolors.Normalize(
        vmin=vmin,
        vmax=vmax,
    )

    fig = plt.figure(
        figsize=(3.0 * len(quantile_surfaces), 3.6)
    )

    for panel, (name, surface) in enumerate(
        quantile_surfaces.items()
    ):
        ax = fig.add_subplot(
            1,
            len(quantile_surfaces),
            panel + 1,
            projection="3d",
        )

        ax.plot_surface(
            M,
            TGRID,
            surface,
            facecolors=cmap(surface_norm(surface)),
            linewidth=0,
            antialiased=True,
            shade=True,
            alpha=0.95,
        )

        ax.set_title(name, pad=1, y=1.01)
        ax.set_xlabel("Moneyness", labelpad=0.5)
        ax.set_ylabel("Tenor (yr)", labelpad=0.5)
        ax.set_zlim(vmin, vmax)
        ax.view_init(elev=25, azim=-55)

        ax.xaxis.set_major_locator(
            mpl.ticker.MaxNLocator(4)
        )
        ax.yaxis.set_major_locator(
            mpl.ticker.MaxNLocator(4)
        )
        ax.zaxis.set_major_locator(
            mpl.ticker.MaxNLocator(4)
        )

    scalar_mappable = cm.ScalarMappable(
        cmap=cmap,
        norm=surface_norm,
    )
    scalar_mappable.set_array([])

    colorbar = fig.colorbar(
        scalar_mappable,
        ax=fig.axes,
        shrink=0.55,
        pad=0.04,
        aspect=25,
    )
    colorbar.set_label("IV (%)")

    fig.legend(
        handles=model_legend_handles,
        loc="lower center",
        ncol=len(model_legend_handles),
        bbox_to_anchor=(0.5, -0.01),
        title="Models",
    )

    fig.suptitle(
        f"{quantile}th-Percentile Implied Volatility Surface",
        y=0.92,
    )

    plt.savefig(
        FIG_DIR / f"Models_{quantile}quantile.png",
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
# Final :
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                         stderr=subprocess.DEVNULL).strip()
except Exception:
    git_commit = None

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_cube": str(CUBE_PATH),
    "source_cube_sha256": sha256(CUBE_PATH),
    "training_mode": "Full retraining only; no pre-existing checkpoint or saved-result loading",
    "comparison_scope": "Core models; no SBJTS or LightSB",
    "seed": SEED,
    "latent_hpo_seeds": list(LATENT_HPO_SEEDS),
    "volgan_seeds": list(VOLGAN_SEEDS),
    "diffusion_seeds": list(DIFFPEN_SEEDS),
    "checkpoint_dependency": "none; checkpoint files are outputs only",
    "epochs": {
        "ae_fm_final": EPOCHS,
        "volgan_per_seed": VOLGAN_N_EPOCHS,
        "diffusion_per_seed": DIFFUSION_EPOCHS,
    },
    "selected_latent_dimension": int(BEST_Z_DIM),
    "n_generated": N_GEN,
    "models": list(models_primary),
    "checkpoints": {
        "AE": str(AE_CKPT),
        "AE_compatibility_alias": str(AE_CKPT_ALIAS),
        "L_FM": str(FM_CKPT),
        "L_FM_compatibility_alias": str(FM_CKPT_ALIAS),
        "Diffusion_best": str(OUT_DIR / "iv_ddpm_best.pt"),
        "Diffusion_best_seed": int(BEST_DP_SEED),
        "VolGAN_per_seed": [
            str(OUT_DIR / f"volgan_seed{seed}.pt") for seed in VOLGAN_SEEDS
        ],
        "VolGAN_best_raw_seed": int(BEST_VG_RAW_SEED),
        "VolGAN_best_reweighted_seed": int(BEST_VG_RW_SEED),
    },
    "latent_dimension_study": {
        "candidates": list(LATENT_DIM_CANDIDATES),
        "seeds": list(LATENT_HPO_SEEDS),
        "ae_epochs_per_trial": int(LATENT_HPO_AE_EPOCHS),
        "fm_epochs_per_trial": int(LATENT_HPO_FM_EPOCHS),
        "n_generated_per_trial": int(LATENT_HPO_N_GEN),
        "n_sliced_wasserstein_projections": int(LATENT_HPO_N_PROJ),
        "legacy_arbitrage_weight": float(LATENT_ARB_WEIGHT),
        "legacy_arbitrage_tolerance": float(LEGACY_ARB_TOL),
        "trial_results_csv": str(LATENT_STUDY_CSV),
        "summary_csv": str(LATENT_SUMMARY_CSV),
    },
    "multiseed_metrics": {
        "lfm_summary_csv": str(OUT_DIR / "lfm_multiseed_summary_from_hpo.csv"),
        "volgan_summary_csv": str(OUT_DIR / "volgan_multiseed_summary.csv"),
        "diffusion_summary_csv": str(OUT_DIR / "diffusion_multiseed_summary.csv"),
        "combined_summary_csv": str(OUT_DIR / "multiseed_summary_all.csv"),
    },
    "figure_policy": {
        "font_family": "Linux Libertine O with Times-style fallbacks",
        "font_size_pt": 9,
        "legends_on_all_figures": True,
        "savefig_dpi": int(mpl.rcParams["savefig.dpi"]),
    },
    "generated_surfaces": str(OUT_DIR / "core_model_surfaces.npz"),
    "figures": sorted(str(p) for p in FIG_DIR.glob("*.png")),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "torch": torch.__version__,
        "device": str(device),
    },
    "git_commit": git_commit,
}
(OUT_DIR / "run_manifest_core.json").write_text(json.dumps(manifest, indent=2))
print("Saved:", OUT_DIR / "run_manifest_core.json")
print("Figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" ", p)


In [ ]:
# -- Count of arbitrage-free surfaces per model ------------------------------
# Reuses legacy_phi_batch / LEGACY_ARB_TOL (defined in the latent-dimension
# HPO cell above) so the arbitrage-free criterion matches zero_arb_pct
# elsewhere in this notebook: a surface is arbitrage-free when |phi| < LEGACY_ARB_TOL.

arb_free_rows = []
for name, arr in models_primary.items():
    phi = legacy_phi_batch(arr.astype(np.float64))
    n_total = len(phi)
    n_free = int(np.sum(np.abs(phi) < LEGACY_ARB_TOL))
    arb_free_rows.append({
        "Model": name,
        "n_surfaces": n_total,
        "n_arbitrage_free": n_free,
        "pct_arbitrage_free": 100.0 * n_free / n_total,
    })

arb_free_df = pd.DataFrame(arb_free_rows)
print(arb_free_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.bar(
    arb_free_df["Model"],
    arb_free_df["pct_arbitrage_free"],
    color=[clr[name] for name in arb_free_df["Model"]],
    alpha=0.85,
)
ax.set_ylabel("Arbitrage-free surfaces (%)")
ax.set_ylim(0, 100)
ax.set_title("Share of Arbitrage-Free Surfaces per Model")
for tick in ax.get_xticklabels():
    tick.set_rotation(15)
plt.tight_layout()
plt.show()
